Makes a summarized `.tsv` file based on chosen atlas and map.

In [35]:
# Import packages
from nilearn import plotting as npl
import SUITPy as suit
import SUITPy.atlas as atlas
import nibabel as nib
import ants
import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path # for saving file name

import pandas as pd

In [ ]:
# tissues: 'gm', 'wm', 'csf', 
# "tissue" is a bad name...think of something more accurate to incl csf, etc.
tissue = 'gm' # default
tissue_dict = {
    'gm': 'c1',
    'wm': 'c2',
    'csf': 'c3'
}

# directories
anat_dir = 'smarts_cerebellum/anatomicals'
p_df = pd.read_csv('smarts_cerebellum/anatomicals/participants_anat.tsv', sep = '\t')

# should I store these results in a new folder or in the old folder?
# maybe inside the anats directory, inside a folder called f'{tissue}_results' so that new folder for each tissue

for i in range(0, p_df.shape[0]):

    p_id = p_df['ID'].iloc[i]
    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
    p_centre = (str(p_df['Centre'].iloc[i])).strip()
    refT1 = p_df['RefT1'].iloc[i]

    subj_id = f'{p_centre.strip()}_{p_id}'

    # path to store results from this pipeline for each participant + timepoint
    results_path = Path(anat_dir)/subj_id/week

    t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
    tissue_path = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1.nii'

    # check that paths exist
    if not Path(t1_path).is_file():
        print(f'T1 path does not exist for {subj_id} in week {week}')
        continue
    if not Path(tissue_path).is_file():
        print(f'{tissue} path does not exist for {subj_id} in week {week}')
        continue

    tissue_vol = f'{results_path}/{subj_id}_{week}_T1_{tissue}_vol.nii'
    tissue_vol_img = nib.load(tissue_vol)
    
    # note that the volume of voxels in this image and the output from reslice are the same, since they're both in SUIT space
    voxel_dim = tissue_vol_img.header.get_zooms()[:3]
    voxel_vol = np.prod(voxel_dim)

    atlas.fetch_atlas('Nettekoven_2024')
    df = atlas.summarize_data(tissue_vol_img,
                              space = 'SUIT',
                              stats = ['nanmean'],
                              atlas = 'Nettekoven_2024',
                              maps = 'atl-NettekovenAsym32'
                              )
    df[f'{tissue}v'] = df['nanmean'] * (df['size']/voxel_vol)

    df.rename(columns={'nanmean': f'avg_{tissue}v'}, inplace = True)
    df['image_name'] = f'{tissue}v_img_{subj_id}_{week}_T1w_LPI.nii'
    df['subj_id'] = subj_id
    df['week'] = week
    df.head(5)

    # put all of this in a new dataframe
    if i == 0:
        # header for only the first run
        all_df = df
    else:
        all_df = pd.concat([all_df, df], ignore_index = True)

    # select the row to write descriptive data to
    row_mask = (all_df['subj_id']==subj_id) & (all_df['week']==week)
    print(f'Writing data for {subj_id} at {week}')

    all_df.loc[row_mask, 'ID'] = p_id
    all_df.loc[row_mask, 'Week'] = week
    all_df.loc[row_mask, 'Centre'] = p_centre
    all_df.loc[row_mask, 'RefT1'] = refT1
    all_df.loc[row_mask, 'age'] = str(p_df['age'].iloc[i])
    all_df.loc[row_mask, 'Gender'] = p_df['Gender'].iloc[i]
    all_df.loc[row_mask, 'isPatient'] = str(p_df['isPatient'].iloc[i])
    all_df.loc[row_mask, 'LesionSide'] = p_df['LesionSide'].iloc[i]
    all_df.loc[row_mask, 'LesionLocation'] = p_df['LesionLocation'].iloc[i]
    all_df.loc[row_mask, 'handedness'] = str(p_df['handedness'].iloc[i])

all_df.to_csv(f'{anat_dir}/{tissue}v_atlas_summarized.tsv', mode = 'w', sep = '\t', index = False, header = True)


0     True
1     True
2     True
3     True
4     True
5     True
6     True
7     True
8     True
9     True
10    True
11    True
12    True
13    True
14    True
15    True
16    True
17    True
18    True
19    True
20    True
21    True
22    True
23    True
24    True
25    True
26    True
27    True
28    True
29    True
30    True
31    True
dtype: bool

Writing descriptive data for CU_2310 in week W4 to row 0


0     False
1     False
2     False
3     False
4     False
      ...  
59     True
60     True
61     True
62     True
63     True
Length: 64, dtype: bool

Writing descriptive data for CU_2310 in week W12 to row 32


0     False
1     False
2     False
3     False
4     False
      ...  
91     True
92     True
93     True
94     True
95     True
Length: 96, dtype: bool

...

Writing descriptive data for CU_2538 in week W4 to row 128
T1 path does not exist for CU_2663 in week W1


0      False
1      False
2      False
3      False
4      False
       ...  
187     True
188     True
189     True
190     True
191     True
Length: 192, dtype: bool

Writing descriptive data for CU_2663 in week W4 to row 160


0      False
1      False
2      False
3      False
4      False
       ...  
219     True
220     True
221     True
222     True
223     True
Length: 224, dtype: bool

Writing descriptive data for CU_2663 in week W12 to row 192


In [54]:
all_df

,image,image_name,frame,region,regionname,size,atlas,map,space,avg_gmv,...,ID,Week,Centre,RefT1,age,Gender,isPatient,LesionSide,LesionLocation,handedness
0,1,gmv_img_CU_2310_W4_T1w_LPI.nii,0,1,M1L,1699.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.565587,...,2310.0,W4,CU,W0,57,M,1,left,subcortical,2
1,1,gmv_img_CU_2310_W4_T1w_LPI.nii,0,2,M2L,4252.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.583018,...,2310.0,W4,CU,W0,57,M,1,left,subcortical,2
2,1,gmv_img_CU_2310_W4_T1w_LPI.nii,0,3,M3L,9115.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.576370,...,2310.0,W4,CU,W0,57,M,1,left,subcortical,2
3,1,gmv_img_CU_2310_W4_T1w_LPI.nii,0,4,M4L,2329.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.548306,...,2310.0,W4,CU,W0,57,M,1,left,subcortical,2
4,1,gmv_img_CU_2310_W4_T1w_LPI.nii,0,5,A1L,717.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.559686,...,2310.0,W4,CU,W0,57,M,1,left,subcortical,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,1,gmv_img_CU_2663_W12_T1w_LPI.nii,0,28,S1R,8289.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.515199,...,2663.0,W12,CU,W0,67,F,1,left,subcortical,2
220,1,gmv_img_CU_2663_W12_T1w_LPI.nii,0,29,S2R,3971.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.543496,...,2663.0,W12,CU,W0,67,F,1,left,subcortical,2
221,1,gmv_img_CU_2663_W12_T1w_LPI.nii,0,30,S3R,6423.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.538598,...,2663.0,W12,CU,W0,67,F,1,left,subcortical,2
222,1,gmv_img_CU_2663_W12_T1w_LPI.nii,0,31,S4R,10162.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.326565,...,2663.0,W12,CU,W0,67,F,1,left,subcortical,2


In [38]:
# tissues: 'gm', 'wm', 'csf', 
# "tissue" is a bad name...think of something more accurate to incl csf, etc.
tissue = 'gm' # default
tissue_dict = {
    'gm': 'c1',
    'wm': 'c2',
    'csf': 'c3'
}

# directories

# OUTDATED PATHS - NEED TO CHANGE; ALSO, OUTDATED TSV
# CHECK PATHS AND TSV
anat_dir = 'backup anats/smarts_cerebellum backup2/anatomicals'
p_df = pd.read_csv('backup anats/smarts_cerebellum backup2/participants.tsv', sep = '\t')

# should I store these results in a new folder or in the old folder?
# maybe inside the anats directory, inside a folder called f'{tissue}_results' so that new folder for each tissue

for i in range(0, p_df.shape[0]):

    p_id = p_df['ID'].iloc[i]
    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
    p_centre = (str(p_df['Centre'].iloc[i])).strip()
    refT1 = p_df['RefT1'].iloc[i]

    subj_id = f'{p_centre.strip()}_{p_id}'

    results_path = Path(anat_dir)/subj_id/week/f'{tissue}_results'

    #t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
    #tissue_path = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1.nii'
    t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1w_LPI.nii'
    tissue_path = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1w_LPI.nii'

    # check that paths exist
    if not Path(t1_path).is_file():
        print(f'T1 path does not exist for {subj_id} in week {week}')
        continue
    if not Path(tissue_path).is_file():
        print(f'{tissue} path does not exist for {subj_id} in week {week}')
        continue

    tissue_vol_img = results_path/f'{subj_id}_{week}_T1_{tissue}_vol.nii'
    
    # note that the volume of voxels in this image and the output from reslice are the same, since they're both in SUIT space
    voxel_dim = tissue_vol_img.header.get_zooms()[:3]
    voxel_vol = np.prod(voxel_dim)

    atlas.fetch_atlas('Nettekoven_2024')
    df = atlas.summarize_data(tissue_vol_img,
                              space = 'SUIT',
                              stats = ['nanmean'],
                              atlas = 'Nettekoven_2024',
                              maps = 'atl-NettekovenAsym32'
                              )
    df[f'{tissue}v'] = df['nanmean'] * (df['size']/voxel_vol)

    df.rename(columns={'nanmean': f'avg_{tissue}v'}, inplace = True)
    df['image_name'] = tissue_vol_img
    df.head(5)

    # put all of this in a new dataframe
    if i == 0:
        # header for only the first run
        all_df = df
    else:
        all_df = pd.concat([all_df, df], ignore_index = True)
    
    # read rows of participants.tsv and get descriptive data for each row
    row_mask = all_df['image_name'] == tissue_vol_img
    all_df.loc[row_mask, 'ID'] = p_id
    all_df.loc[row_mask, 'Week'] = week
    all_df.loc[row_mask, 'Centre'] = p_centre
    all_df.loc[row_mask, 'RefT1'] = refT1

    all_df.loc[row_mask, 'age'] = str(p_df['age'].iloc[i])
    all_df.loc[row_mask, 'Gender'] = p_df['Gender'].iloc[i]
    all_df.loc[row_mask, 'isPatient'] = str(p_df['isPatient'].iloc[i])
    all_df.loc[row_mask, 'LesionSide'] = p_df['LesionSide'].iloc[i]
    all_df.loc[row_mask, 'LesionLocation'] = p_df['LesionLocation'].iloc[i]
    all_df.loc[row_mask, 'handedness'] = str(p_df['handedness'].iloc[i])

all_df.to_csv(f'{anat_dir}/{tissue}v_atlas_summarized.tsv', mode = 'w', sep = '\t', index = False, header = True)


AttributeError: 'PosixPath' object has no attribute 'header'